In [ ]:
!pip install albumentations==1.4.3 opencv-python-headless==4.10.0.84 segmentation-models-pytorch kagglehub
!pip install -q kaggle

import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch import nn
import os
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from google.colab import userdata
import kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.0/137.0 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.6 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: albumentations
    Found existing installation: albumentations 2.0.8
    Uninstalling albumentations-2.0.8:
      Successfully uninstalled albumentations-2.0.8


In [ ]:
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')


In [ ]:
!kaggle datasets download -d "franzwagner/river-water-segmentation-dataset"

Dataset URL: https://www.kaggle.com/datasets/franzwagner/river-water-segmentation-dataset
License(s): Attribution-NonCommercial 4.0 International (CC BY-NC 4.0)
 92% 620M/672M [00:03<00:00, 63.0MB/s]
100% 672M/672M [00:03<00:00, 189MB/s] 


In [ ]:
!unzip "river-water-segmentation-dataset"

Archive:  river-water-segmentation-dataset.zip
  inflating: riwa_v2/images/ADE_frame_00000044.jpg  
  inflating: riwa_v2/images/ADE_frame_00000077.jpg  
  inflating: riwa_v2/images/ADE_frame_00000078.jpg  
  inflating: riwa_v2/images/ADE_frame_00000081.jpg  
  inflating: riwa_v2/images/ADE_frame_00000112.jpg  
  inflating: riwa_v2/images/ADE_frame_00000114.jpg  
  inflating: riwa_v2/images/ADE_frame_00000122.jpg  
  inflating: riwa_v2/images/ADE_frame_00000123.jpg  
  inflating: riwa_v2/images/ADE_frame_00000126.jpg  
  inflating: riwa_v2/images/ADE_frame_00000138.jpg  
  inflating: riwa_v2/images/ADE_frame_00000158.jpg  
  inflating: riwa_v2/images/ADE_frame_00000197.jpg  
  inflating: riwa_v2/images/ADE_frame_00000198.jpg  
  inflating: riwa_v2/images/ADE_frame_00000200.jpg  
  inflating: riwa_v2/images/ADE_frame_00000225.jpg  
  inflating: riwa_v2/images/ADE_frame_00000252.jpg  
  inflating: riwa_v2/images/ADE_frame_00000256.jpg  
  inflating: riwa_v2/images/ADE_frame_00000257.jpg  

In [ ]:
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset

VALID_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp')

class RiverDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform

        self.images = [f for f in os.listdir(image_dir) if f.lower().endswith(VALID_EXTENSIONS)]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.image_dir, img_name)

        base_name, _ = os.path.splitext(img_name)
        mask_name = base_name + ".png"
        mask_path = os.path.join(self.mask_dir, mask_name)

        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f"Image file not found or is unreadable at: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # Convert to RGB

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise FileNotFoundError(f"Mask file not found or is unreadable at: {mask_path}")

        _, mask = cv2.threshold(mask, 128, 255, cv2.THRESH_BINARY)

        mask = mask / 255.0

        mask = np.expand_dims(mask, axis=-1)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        return image, mask

In [ ]:
from google.colab import files
# uploaded = files.upload()  # select unet_river.pth

# device = "cuda" if torch.cuda.is_available() else "cpu"
# model = UNet().to(device)
# model.load_state_dict(torch.load("unet_river.pth", map_location=device))
# model.eval()

import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
)

model.to(device)

print("carregado")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

carregado


In [ ]:
# --- Define Transforms ---
IMG_MEAN = [0.485, 0.456, 0.406]
IMG_STD = [0.229, 0.224, 0.225]
IMG_HEIGHT, IMG_WIDTH = 256, 256

train_transform = A.Compose([
    A.Resize(height=IMG_HEIGHT, width=IMG_WIDTH),
    A.Rotate(limit=35, p=0.3),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.1),
    A.Normalize(mean=IMG_MEAN, std=IMG_STD, max_pixel_value=255.0),
    ToTensorV2(transpose_mask=True), # Correctly transposes mask
])

val_transform = A.Compose([
    A.Resize(height=IMG_HEIGHT, width=IMG_WIDTH),
    A.Normalize(mean=IMG_MEAN, std=IMG_STD, max_pixel_value=255.0),
    ToTensorV2(transpose_mask=True),
])

# --- Define dataset root path ---
# 'path' variable comes from your download cell (Cell 2)
dataset_root = os.path.join("riwa_v2")

TRAIN_IMG_DIR = os.path.join(dataset_root, "images")
TRAIN_MASK_DIR = os.path.join(dataset_root, "masks")
VAL_IMG_DIR = os.path.join(dataset_root, "validation", "images")
VAL_MASK_DIR = os.path.join(dataset_root, "validation", "masks")

# --- Create Datasets and DataLoaders ---
train_dataset = RiverDataset(
    image_dir=TRAIN_IMG_DIR,
    mask_dir=TRAIN_MASK_DIR,
    transform=train_transform
)

# Now we create the validation dataset
val_dataset = RiverDataset(
    image_dir=VAL_IMG_DIR,
    mask_dir=VAL_MASK_DIR,
    transform=val_transform
)

# Create the DataLoaders
BATCH_SIZE = 8
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

# --- Test if it works ---
print("Testing DataLoaders...")
images, masks = next(iter(train_loader))
print(f"Image batch shape: {images.shape}")
print(f"Mask batch shape: {masks.shape}")

images_val, masks_val = next(iter(val_loader))
print(f"Val Image batch shape: {images_val.shape}")
print(f"Val Mask batch shape: {masks_val.shape}")
print("✅ Train and Validation DataLoaders created.")

Testing DataLoaders...
Image batch shape: torch.Size([8, 3, 256, 256])
Mask batch shape: torch.Size([8, 1, 256, 256])
Val Image batch shape: torch.Size([8, 3, 256, 256])
Val Mask batch shape: torch.Size([8, 1, 256, 256])
✅ Train and Validation DataLoaders created.


In [ ]:
import torch.optim as optim
import segmentation_models_pytorch as smp
from tqdm import tqdm # For a nice progress bar

# --- Parameters ---
LEARNING_RATE = 1e-4
NUM_EPOCHS = 30 # Start with 25, you can increase this if needed
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Define Model ---
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
)
model.to(DEVICE)

# --- Loss Function & Optimizer ---
loss_fn = smp.losses.DiceLoss(mode='binary', from_logits=True)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scaler = torch.cuda.amp.GradScaler() # For mixed precision (faster training)

# --- Main Training & Validation Loop ---
print("Starting training...")
best_val_loss = float('inf') # Save the model with the best *validation* loss

for epoch in range(NUM_EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{NUM_EPOCHS} ---")

    # --- Training Phase ---
    model.train()
    train_loss = 0.0
    train_loop = tqdm(train_loader, desc=f"Train Epoch {epoch+1}", leave=False)

    for batch_idx, (data, targets) in enumerate(train_loop):
        data = data.to(device=DEVICE)
        targets = targets.to(device=DEVICE, dtype=torch.float32)

        with torch.cuda.amp.autocast():
            predictions = model(data)
            loss = loss_fn(predictions, targets)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        train_loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / len(train_loader)
    print(f"Epoch {epoch+1} Avg Train Loss: {avg_train_loss:.4f}")

    # --- Validation Phase ---
    model.eval()
    val_loss = 0.0
    val_loop = tqdm(val_loader, desc=f"Valid Epoch {epoch+1}", leave=False)

    with torch.no_grad():
        for data, targets in val_loop:
            data = data.to(device=DEVICE)
            targets = targets.to(device=DEVICE, dtype=torch.float32)

            with torch.cuda.amp.autocast():
                predictions = model(data)
                loss_val = loss_fn(predictions, targets)

            val_loss += loss_val.item()
            val_loop.set_postfix(val_loss=loss_val.item())

    avg_val_loss = val_loss / len(val_loader)
    print(f"Epoch {epoch+1} Avg Val Loss:   {avg_val_loss:.4f}")

    # --- Save the best model based on validation loss ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "best_model.pth")
        print(f"==> Validation Loss improved! Saving new best_model.pth")

print("\n🎉 Training finished!")

/tmp/ipython-input-712531323.py:22: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() # For mixed precision (faster training)


Starting training...

--- Epoch 1/30 ---


Train Epoch 1:   0%|          | 0/143 [00:00<?, ?it/s]/tmp/ipython-input-712531323.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1 Avg Train Loss: 0.2207


Valid Epoch 1:   0%|          | 0/21 [00:00<?, ?it/s]/tmp/ipython-input-712531323.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


...

Epoch 28 Avg Val Loss:   0.0326
==> Validation Loss improved! Saving new best_model.pth

--- Epoch 29/30 ---


Epoch 29 Avg Train Loss: 0.0300


Epoch 29 Avg Val Loss:   0.0369

--- Epoch 30/30 ---


Epoch 30 Avg Train Loss: 0.0282


Epoch 30 Avg Val Loss:   0.0373

🎉 Training finished!


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A

inference_transform = A.Compose([
    A.Resize(height=IMG_HEIGHT, width=IMG_WIDTH),
    A.Normalize(mean=IMG_MEAN, std=IMG_STD, max_pixel_value=255.0),
    ToTensorV2(),
])

# --- 2. Helper Function: Get Mask from Image ---
def get_water_mask(image_path, model):
    # Load Image
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"Could not find image at {image_path}")

    original_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    original_h, original_w = original_image.shape[:2]

    # Preprocess
    augmented = inference_transform(image=original_image)
    img_tensor = augmented["image"].unsqueeze(0).to(device)

    # Predict
    model.eval()
    with torch.no_grad():
        logits = model(img_tensor)
        pred_prob = torch.sigmoid(logits)[0, 0].cpu().numpy()

    pred_resized = cv2.resize(pred_prob, (original_w, original_h))

    # --- THE FIX IS HERE ---
    # Your model thinks 1=Land and 0=Water.
    # We want 1=Water.
    # So we take everything where the model is "unsure" or predicts 0 (Low prob)
    # and call THAT water.

    # If model output < 0.5 (It thinks it's NOT land), we mark it as Water (1)
    binary_mask = (pred_resized < 0.5).astype(np.uint8)

    return original_image, binary_mask

# --- 3. Main Comparison Logic ---
def analyze_river_levels(benchmark_path, current_path, model, threshold_sensitivity=0.05):
    """
    Compares two images to detect flood/drought.
    Automatically handles resolution mismatches and prevents integer overflow.
    """
    print(f"Processing Benchmark: {benchmark_path}...")
    img_ref, mask_ref = get_water_mask(benchmark_path, model)

    print(f"Processing Current: {current_path}...")
    img_cur, mask_cur = get_water_mask(current_path, model)

    # --- FIX 1: Normalize Resolutions ---
    # We resize the 'current' image/mask to match the 'benchmark' dimensions
    # so we can compare them pixel-by-pixel.
    if img_ref.shape[:2] != img_cur.shape[:2]:
        print(f"Warning: Resizing current image from {img_cur.shape[:2]} to {img_ref.shape[:2]} for comparison.")
        target_h, target_w = img_ref.shape[:2]
        # Resize image for display
        img_cur = cv2.resize(img_cur, (target_w, target_h))
        # Resize mask for calculation (use nearest neighbor to keep it binary 0/1)
        mask_cur = cv2.resize(mask_cur, (target_w, target_h), interpolation=cv2.INTER_NEAREST)

    # --- FIX 2: Float Conversion for Math ---
    # Convert to float to avoid integer overflow/underflow during subtraction
    water_pixels_ref = float(np.sum(mask_ref))
    water_pixels_cur = float(np.sum(mask_cur))

    total_pixels = mask_ref.size
    coverage_ref = water_pixels_ref / total_pixels
    coverage_cur = water_pixels_cur / total_pixels

    if water_pixels_ref == 0:
        print("Error: No water detected in benchmark image (0 pixels). Cannot calculate percentage change.")
        return

    # Calculate Change
    pct_change = (water_pixels_cur - water_pixels_ref) / water_pixels_ref

    # --- Determine Status ---
    status = "NORMAL"

    if pct_change > threshold_sensitivity:
        status = "FLOOD WARNING (Cheia)"
    elif pct_change < -threshold_sensitivity:
        status = "DROUGHT WARNING (Seca)"

    # --- Visualization ---
    print("\n" + "="*40)
    print(f"RESULT: {status}")
    print(f"Water Coverage Change: {pct_change*100:.2f}%")
    print("="*40 + "\n")

    fig, ax = plt.subplots(1, 3, figsize=(18, 6))

    # Show Benchmark
    ax[0].imshow(img_ref)
    ax[0].imshow(mask_ref, cmap='Blues', alpha=0.4)
    ax[0].set_title(f"referência: {coverage_ref*100:.1f}%")
    ax[0].axis('off')

    # Show Current
    ax[1].imshow(img_cur)
    ax[1].imshow(mask_cur, cmap='Blues', alpha=0.4)
    ax[1].set_title(f"atual: {coverage_cur*100:.1f}%")
    ax[1].axis('off')

    # Show Difference Map
    diff_map = np.zeros((*mask_ref.shape, 3), dtype=np.uint8)

    # Logical comparisons (ensure masks are boolean 0/1 or similar)
    # We use > 0 just to be safe against any resizing artifacts
    ref_bool = mask_ref > 0
    cur_bool = mask_cur > 0

    # Water in BOTH = Green (Stable)
    diff_map[ref_bool & cur_bool] = [0, 255, 0]
    # Water in Current ONLY = Red (Flood)
    diff_map[~ref_bool & cur_bool] = [255, 0, 0]
    # Water in Benchmark ONLY = Yellow (Receded/Drought)
    diff_map[ref_bool & ~cur_bool] = [255, 255, 0]

    ax[2].imshow(diff_map)
    ax[2].set_title(f"análise de mudança \n vermelho=cheia, amarelho=seca, verde=estável")
    ax[2].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
from google.colab import files

# Load the best model from training
# Ensure 'best_model.pth' exists from the training loop
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=3,
    classes=1,
)
model.load_state_dict(torch.load("best_model.pth"))
model.to(device)
print("Model loaded successfully.")

# --- INTERFACE ---
# Replace these strings with the actual filenames you uploaded to Colab
benchmark_file = "rio_normal.jpg" # <--- PUT YOUR NORMAL IMAGE NAME HERE
current_file = "rio_flood.png"    # <--- PUT YOUR TEST IMAGE NAME HERE



# Check if files exist before running
import os
if os.path.exists(benchmark_file) and os.path.exists(current_file):
    analyze_river_levels(benchmark_file, current_file, model)
else:
    print("Please upload the images and update the filenames in the code above.")

NameError: name 'smp' is not defined